In [10]:
prompt="""Solve the following math problem.

"Let $a, b, c$ be distinct numbers such that the equations $x^2 + ax + 1 = 0$ and $x^2 + bx + c = 0$ have a common real root, and the equations $x^2 + x + a = 0$ and $x^2 + cx + b = 0$ also have a common real root. Compute the sum $a + b + c$.,


You must output only the final answer.
Answer:"""

In [8]:
import requests, math
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("/home/ivaning/models/gpt-oss-20b")

def score_answer_prompt_logprobs(prompt, answer, port=8006, model="gpt-oss-20b"):
    full = prompt + answer
    ctx_ids = tok.encode(prompt, add_special_tokens=False)
    full_ids = tok.encode(full, add_special_tokens=False)
    start = len(ctx_ids)

    r = requests.post(
        f"http://127.0.0.1:{port}/v1/completions",
        json={
            "model": model,
            "prompt": full,
            "max_tokens": 1,
            "temperature": 0,
            "prompt_logprobs": 1,
        },
        timeout=120,
    )
    r.raise_for_status()
    plps = r.json()["choices"][0]["prompt_logprobs"]

    token_logps = []
    for pos in range(start, len(full_ids)):
        tid = full_ids[pos]
        entry = plps[pos]
        obj = entry.get(str(tid)) or entry.get(tid)
        lp = obj["logprob"] if isinstance(obj, dict) else obj.logprob
        token_logps.append(lp)

    logp = sum(token_logps)
    return logp, math.exp(logp), token_logps



In [5]:
score = score_answer_prompt_logprobs(prompt, " -3", port=8006)
score

(-12.194347858428955,
 5.058968959212135e-06,
 [-8.638436317443848, -3.5559115409851074])

In [11]:
import re

r = requests.post(
    "http://127.0.0.1:8006/v1/chat/completions",
    json={
        "model": "gpt-oss-20b",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,
        "max_tokens": 8192,
    },
    timeout=300,
)
msg = r.json()["choices"][0]["message"]
reason = (msg.get("reasoning") or "").strip()
answer = (msg.get("content") or "").strip()
print("rollout answer:", answer)
print("reasoning tail:", reason[-80:])

matches = list(re.finditer(r"(?:Final )?Answer:\s*", reason, re.I))
prefix = reason[: matches[-1].end()] if matches else reason + "\nAnswer: "

print("at prompt:", score_answer_prompt_logprobs(prompt, answer or "-3"))
print("at rollout:", score_answer_prompt_logprobs(prefix, answer or "-3"))

rollout answer: -3
reasoning tail:  x^2 + c x + b = x^2 -1x +0 = x(x-1). Root 1. Works. So sum -3.

Thus answer -3.
at prompt: (-6.1938276290893555, 0.0020419957733713028, [-6.1938276290893555])
at rollout: (-0.00012885693286079913, 0.9998711513688372, [-0.00012885693286079913])


In [ ]:
import re

gold, wrongs = "-3", ["3", "0", "1", "-1"]
cands = [gold] + wrongs
problem = (
    "Let $a, b, c$ be distinct numbers such that the equations $x^2 + ax + 1 = 0$ "
    "and $x^2 + bx + c = 0$ have a common real root, and the equations $x^2 + x + a = 0$ "
    "and $x^2 + cx + b = 0$ also have a common real root. Compute the sum $a + b + c$."
)
hints = {
    "baseline": "",
    "good": (
        "Set r and s as the shared roots. Eliminate a and b from the four root equations. "
        "Distinctness rules out the c=1 branch, so the real branch forces s=1; "
        "then compute a and b+c from the root equations, and finally sum a+b+c."
    ),
    "bad": (
        "Assume c=1 first and test small integer roots in {0,1,-1}. "
        "The sum a+b+c is usually positive for this type of problem."
    ),
}

def make_prompt(hint=""):
    p = f"Solve the following math problem.\n\n{problem}\n"
    if hint:
        p += f"\nAdditional guidance:\n{hint}\n"
    p += "\nYou must output only the final answer.\nAnswer:"
    return p

def early_log_odds(prompt):
    lps = [score_answer_prompt_logprobs(prompt, a)[0] for a in cands]
    lse = math.log(sum(math.exp(x) for x in lps))
    return lps[0] - lse, lps[0]

def em_rollout(prompt):
    r = requests.post(
        "http://127.0.0.1:8006/v1/chat/completions",
        json={"model": "gpt-oss-20b", "messages": [{"role": "user", "content": prompt}],
              "temperature": 0, "max_tokens": 8192},
        timeout=300,
    )
    msg = r.json()["choices"][0]["message"]
    pred = (msg.get("content") or "").strip()
    if not pred:
        m = re.findall(r"(?:Final )?Answer:\s*(\S+)", msg.get("reasoning") or "", re.I)
        pred = m[-1] if m else ""
    pred = pred.rstrip(".")
    return pred, int(pred == gold)

print(f"{'variant':<10} {'log_odds':>9} {'logP(gold)':>10} {'EM':>4} pred")
for name, hint in hints.items():
    p = make_prompt(hint)
    lo, lg = early_log_odds(p)
    pred, em = em_rollout(p)
    print(f"{name:<10} {lo:9.3f} {lg:10.3f} {em:4d} {pred}")